# Introduction to Performance Evaluation, and Factor Models
## 🎯 Learning Objectives

By the end of today you will be able to:

1. **Choose the right performance ratio for the question** — Sharpe, information,
   or appraisal — and say what each one assumes
2. **Run a factor regression** and read α, β, and R² correctly
3. **Split a return into what you could have bought cheaply and what you couldn't**
4. **Compute an appraisal ratio**, and show that Sharpe, information and
   appraisal are one formula against three different benchmarks
5. **Build a hedged portfolio** and explain why hedging buys you alpha capacity
6. **Take apart the most famous track record in finance** — was Berkshire skill,
   or beta? — and see what the length of a record does to its t-statistic

## 📋 Today's Plan

1. [Three questions, three ratios](#three)
2. [Sharpe, the information ratio, and endogenous benchmarks](#sharpe-ir)
3. [Pitfall checklist](#pitfalls)
4. [🔄 Live Demo: was Berkshire skill or beta?](#demo)
5. [Decomposing risk, hedging, and the risk budget](#decomp)
6. [The appraisal ratio — and why there was only ever one ratio](#appraisal)
7. [🛠️ Hands-On: does your signal have alpha?](#ho1)
8. [🎯 Challenge: Exxon vs Pfizer](#challenge) — *homework*
9. [Key takeaways](#takeaways)

---

## 🛠️ Setup

In [1]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4.5]
import warnings; warnings.filterwarnings('ignore')
import pandas_datareader.data as web

BASE  = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"
panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")

ff = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-01-01')[0] / 100
#Below we are converting the index to a timestamp and then adding a MonthEnd offset to ensure that the dates are at the end of each month.
# this is due to the fact that the Fama-French data is of the end of each month, but the index is not timestamp. uncomment below to see what I mean
#print(ff)

ff.index = pd.to_datetime(ff.index.to_timestamp()) + pd.offsets.MonthEnd(0)
ff = ff.loc['1980-01-31':'2000-12-31']

# Berkshire Hathaway, monthly, with the factors on exactly the same months.
# This file carries its own Mkt-RF and RF, so today's regressions use them
# rather than the 1980-2000 series above.
funds = pd.read_pickle('https://raw.githubusercontent.com/amoreira2/Fin418/'
                       'main/assets/data/df_WarrenBAndCathieW_monthly.pkl')
funds.columns = [c.strip() for c in funds.columns]      # 'Mom   ' has trailing spaces
brk = funds.dropna(subset=['BRK'])

print(f"Berkshire: {len(brk)} months, {brk.index.min():%Y-%m} to {brk.index.max():%Y-%m}")
print(f"panel: {len(panel):,} rows")
print(f"factors: {ff.index.min().date()} to {ff.index.max().date()}, "
      f"columns {list(ff.columns)}")

Berkshire: 279 months, 1988-11 to 2021-12
panel: 1,517,174 rows
factors: 1980-01-31 to 2000-12-31, columns ['Mkt-RF', 'SMB', 'HML', 'RF']


---

## 1. Three Questions, Three Ratios <a id="three"></a>

You have a return series. Is it any good?

That is not one question. It is three, and they have different answers.

| Question | Ratio |
|---|---|
| *What is the bang for risk on a univariate basis?* | **Sharpe** |
| *What is the bang for risk relative to a benchmark?* | **Information ratio** |
| *How much it can increase the Sharpe Ratio relative to another asset/assets/factors?* | **Appraisal ratio** |

The third one is the hardest and the most important, and it needs a tool we
don't have yet — a **factor model**. We'll do the first two now, build the
factor model, then come back for the third.

---

## 2. Sharpe, the Information Ratio, and Endogenous Benchmarks <a id="sharpe-ir"></a>

### Sharpe: return per unit of total risk

$$SR = \frac{\text{mean}(r)}{\text{sd}(r)} \times \sqrt{12}$$

For a **long-short** strategy there is no risk-free rate to subtract — the
weights sum to zero, so no capital was tied up and there is nothing to compare
against a T-bill. For a **long-only** position you use the excess return
$r - r_f$.

### Information ratio: return relative to a benchmark

Most money is not managed against cash. It is managed against a **mandate**.

$$IR = \frac{\text{mean}(r_p - r_b)}{\text{sd}(r_p - r_b)} \times \sqrt{12}$$

The numerator is **active return**; the denominator is **tracking error**. No
regression needed — you only have to decide what $r_b$ is.

That decision is not a technical one. Let's see how much it matters.

In [13]:
# A long-only value fund: top BM decile, NYSE breakpoints, value-weighted
panel['me_l1'] = panel.groupby('permno')['me'].shift(1)   # Lecture 2 convention

bm = pd.read_parquet(f"{BASE}/signals/BM.parquet")
dv = panel.merge(bm, on=['permno','date'], how='left').sort_values(['permno','date'])
dv['BM_l1'] = dv.groupby('permno')['BM'].shift(1)
dv = dv.dropna(subset=['BM_l1', 'ret', 'me_l1'])
hi = dv[dv.exchcd == 1].groupby('date')['BM_l1'].quantile(.9).rename('hi')
dv = dv.merge(hi, on='date')
fund = (dv[dv.BM_l1 >= dv.hi].groupby('date')
          .apply(lambda g: np.average(g['ret'], weights=g['me_l1'])))

d0 = panel.dropna(subset=['ret', 'me_l1'])
mkt_vw = d0.groupby('date').apply(lambda g: np.average(g['ret'], weights=g['me_l1']))
mkt_ew = d0.groupby('date')['ret'].mean()
rf     = ff['RF'].reindex(mkt_vw.index)

def info_ratio(p, b):
    a = (p - b).dropna()
    return a.mean()*12, a.std()*np.sqrt(12), a.mean()/a.std()*np.sqrt(12)

print(f"Value fund: {fund.mean()*12:.2%}/yr, vol {fund.std()*np.sqrt(12):.2%}\n")
print(f"{'benchmark':24s}{'active ret':>12s}{'tracking err':>14s}{'IR':>8s}")
print("-"*58)
for name, b in [('VW market', mkt_vw), ('EW market', mkt_ew), ('cash (risk-free)', rf)]:
    a, te, i = info_ratio(fund, b)
    print(f"{name:24s}{a:>11.2%}{te:>14.2%}{i:>8.2f}")

Value fund: 20.85%/yr, vol 16.97%

benchmark                 active ret  tracking err      IR
----------------------------------------------------------
VW market                     5.12%         9.25%    0.55
EW market                     6.65%        11.29%    0.59
cash (risk-free)             14.22%        17.07%    0.83


### One fund. Three numbers.

The same portfolio, the same 251 months, scores **0.55**, **0.59**, or **0.83**
depending only on what you compare it to.

> **💡 Key Insight: the Sharpe ratio is the information ratio against cash**
>
> Look at the last row. When the benchmark is the risk-free rate, active return
> becomes $r_p - r_f$ and tracking error becomes the volatility of the excess
> return — which is the Sharpe ratio exactly. They are not two concepts. Sharpe
> is the special case where your mandate is "don't lose to a T-bill."

### Which benchmark is right?

Whichever one the manager was hired against. That is a contractual question:

| Mandate | Benchmark |
|---|---|
| US large-cap equity fund | S&P 500 |
| Value manager | Russell 1000 Value — *not* the S&P, or they get credit for the value tilt itself |
| Balanced / pension fund | 60% equities, 40% bonds |
| Liability-driven pension | Long-duration government bonds |
| Hedge fund, absolute return | ? |

> **⚠️ Caution: benchmark choice is where performance gets manufactured**
>
> A value manager benchmarked against the *broad market* gets paid for simply
> being a value manager — the whole style tilt shows up as active return. The
> same manager benchmarked against a *value index* has to beat other value
> managers.
>
> It is an art in the mutual fund industry to carefully pick poorly performing benchmarks.
>
> Our fund's IR against the market is 0.55. Against a proper value benchmark it
> would be far lower, because most of that 5% active return **is** the value
> tilt.

That last point is the opening for today. "Most of the active return is the
value tilt" is a claim about *decomposition*, and a benchmark comparison can't
make it precisely. A regression can.



### Or: let the data build the benchmark

There is a third option nobody offers you in a mandate. Instead of picking a
benchmark off a list, **construct the combination of factors that best
replicates the fund**, and use that:

$$r^b_t = \sum_j \beta_j f_{j,t}$$

This is an **endogenous benchmark** — the fitted value of a factor regression.
Whatever mix of market, size and value the manager was actually running, the
regression finds it, and you measure them against that mix rather than against
a label.

We will get to this multi-factor model in a few classes--for today--we will pick one factor.

$$r^b_t = \beta f_{mkt,t}$$

What this bechmark accounts for that $r^b_t = f_{mkt,t}$ does not?

---

## 🛡️ Pitfall Checklist for Factor Regressions <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---|---|---|
| 1 | **Total returns on the left, excess on the right** | α absorbs the risk-free rate and is badly biased | Did you subtract `RF` from the stock's return? |
| 2 | **No intercept** | `sm.OLS(y, X)` without `add_constant` forces α = 0 and biases β | Is there a `const` row in the summary? |
| 3 | **Mixed frequencies** | Monthly returns on daily factors | Do both sides have the same number of rows? |
| 4 | **Reading a high R²** | R² measures how much the factor explains from realized returns | R² near 1 means you measure risk very well |
| 5 | **Annualizing α wrong** | α is per-period; ×12 for monthly, not ×√12 | Volatility scales by √12, means by 12 |
| 6 | **Trusting α without its t-stat** | A big α on a short sample is noise | Is |t| > 2? Over how many months? |



---

## 🔄 Live Demo: Was Berkshire Skill, or Beta? <a id="demo"></a>

Warren Buffett's Berkshire Hathaway is the most famous track record in finance.
From November 1988 to December 2021 it earned **12.5% a year** above T-bills,
carrying **19.5%** volatility — a Sharpe ratio of **0.64**.

The market, over exactly the same months: **0.65**.

On the Sharpe ratio alone, the most celebrated investor of the century is the
index. That cannot be the whole story, and Sharpe has no way to tell you what
else is in there. A regression does.

### First, why there is anything to decompose

Before the algebra, the fact that makes it work. **Stocks do not move
independently.** They co-move, and the degree of co-movement varies in ways you
can more or less guess in advance:

- **Defensive** businesses — utilities, groceries, household staples — barely
  notice the market. People buy electricity and bread in a recession.
- **Cyclical** businesses — luxury goods, banks, airlines, homebuilders — swing
  harder than the market does.
- **Levered** firms swing harder still, because fixed debt payments amplify
  whatever happens to revenue.

This matters before any regression is run. It makes no sense to claim you are
diversified because you hold a lot of different assets, if all of them behave
like each other. So the first question about any set of assets is: **how alike
are they, and along what dimension?** A factor model is the answer written down.

### The decomposition

$$r^e_t = \alpha + \beta \, r^e_{m,t} + \varepsilon_t$$

| Term | Meaning |
|---|---|
| $\beta \, r^e_m$ | The part you could have had by holding the market with leverage — cheap |
| $\alpha$ | Average return **not** explained by market exposure |
| $\varepsilon$ | Idiosyncratic noise — diversifiable, so nobody pays you for it |

> **📌 Remember: the decomposition is always valid**
>
> It is just statistics. Regress anything on anything and you get an intercept,
> a slope and a residual. Nothing has been assumed and nothing has been tested.
> **All of the power is in the interpretation** — in the claim that $\beta r^e_m$
> is something you could have bought for five basis points and $\alpha$ is
> something you couldn't.


### What it says about expected returns

Take expectations of both sides and the residual drops out:

$$E[r] = r_f + \alpha + \beta \, E[f]$$

Three sources, and only three. The time value of money, the premium you earned
for carrying factor exposure, and whatever is left.

> **⚠️ Caution: read α ≠ 0 both ways**
>
> If your model of expected returns is right, α should be **zero** — the factors
> already account for everything. So a significant α means one of exactly two
> things, and the regression cannot tell you which:
>
> 1. You found skill or a mispricing, **or**
> 2. Your factor model is missing something.
>
> A trader usually takes the factor model as given and tries to beat it. If you
> are running a family office, advising a sovereign fund, or setting a mandate,
> deciding *which factors belong in the model* is your job — and then α means
> whatever your choice of model made it mean. We come back to this repeatedly.

### The same equation does two different jobs

Worth separating these, because they get conflated constantly:

| | **Risk model** | **Expected-return model** |
|---|---|---|
| Uses | $\beta$ and $\sigma_\varepsilon$ | $\alpha$ |
| Asks | how much will this move, and with what? | will this earn anything? |
| Concerns the | *second* moment | *first* moment |
| Used by | risk managers, portfolio construction | signal researchers, allocators |

> **💡 Key Insight: alpha is scarce, beta is plentiful — pay different prices**
>
> This is how large allocators actually think. Beta exposure is available for a
> few basis points in an ETF, so nobody should pay a performance fee for it.
> Alpha is hard to find and worth paying for. The whole job is telling them
> apart, and a factor regression is the tool that does it.
>
> The gains from beta come from **implementation** — getting it cheaply. The
> gains from alpha come from **selection** — finding someone who has it.


A model can be excellent at one job and useless at the other. The market factor
explains 22% of Berkshire's variance — a thin risk model, since most of what
Berkshire did had nothing to do with the index. It says nothing at all about
whether Berkshire will earn anything next year.

> **📌 Remember**
>
> Never say "the model works." Say which job it is doing. A high R² is good news
> for the risk model and *bad* news for the alpha hunter, because it means less
> is left over.



### Now ask for it

You have Berkshire's returns in `brk['BRK']`, and the market and the risk-free
rate on the same months in the same table. You want α, β and R².



In [14]:
#MY_PROMPT = "You have Berkshire's returns in brk['BRK'], and the market and the risk-free rate on the same months in the same table. You want α, β and R²."

# ---- paste the AI's code below ----

# 计算 Berkshire 的超额收益 (Excess Return)
brk_excess = brk['BRK'] - brk['RF']

# 提取市场超额收益
mkt_excess = brk['Mkt-RF']

# 添加常数项以计算截距 (Alpha)
X = sm.add_constant(mkt_excess)

# 运行 OLS 回归
model = sm.OLS(brk_excess, X).fit()

# 提取结果 (Alpha 需要年化，即乘以 12)
alpha = model.params['const'] * 12
beta = model.params['Mkt-RF']
r_squared = model.rsquared

print(f"Alpha (年化): {alpha:.2%}")
print(f"Beta: {beta:.2f}")
print(f"R²: {r_squared:.2f}")

Alpha (年化): 6.54%
Beta: 0.59
R²: 0.22


In [4]:
#@title 🔒 Check — run after you've pasted yours
# The same regression, specified three ways.
tot = brk['BRK']                     # total return
ex  = brk['BRK'] - brk['RF']         # excess return
Xm  = brk['Mkt-RF']

runs = {'excess returns, with intercept   (correct)': sm.OLS(ex, sm.add_constant(Xm)).fit(),
        'TOTAL returns, with intercept    (pitfall 1)': sm.OLS(tot, sm.add_constant(Xm)).fit(),
        'excess returns, NO intercept     (pitfall 2)': sm.OLS(ex, Xm).fit()}

print(f"{'specification':46s}{'beta':>8s}{'alpha/yr':>11s}")
print("-"*65)
for k, r in runs.items():
    al = r.params['const']*12 if 'const' in r.params else 0.0
    print(f"{k:46s}{r.params['Mkt-RF']:>8.2f}{al:>10.2%}" + ("  (forced)" if 'const' not in r.params else ""))
print(f"\n  mean risk-free rate over the sample: {brk['RF'].mean()*12:.2%}/yr")

specification                                     beta   alpha/yr
-----------------------------------------------------------------
excess returns, with intercept   (correct)        0.59     6.54%
TOTAL returns, with intercept    (pitfall 1)      0.59     9.36%
excess returns, NO intercept     (pitfall 2)      0.61     0.00%  (forced)

  mean risk-free rate over the sample: 2.80%/yr


### The one that bites

Dropping the intercept barely moves β here — 0.59 to 0.61. Worth knowing, not
fatal.

**Forgetting to subtract the risk-free rate hands the whole risk-free rate to
alpha**, taking it from **+6.5%** to **+9.4%** a year. The risk-free rate
averaged 2.8%/yr over these months, and with nothing else to absorb it, α
swallowed it whole. Run the same mistake on a 1980s sample, where T-bills paid
three times as much, and the damage is far worse.

>
> Neither version errors. Both print a beta near 0.59 and an R² near 0.22, so the
> output *looks* the same. The only defence is having said "excess returns" in
> the request — which is why the specification is the skill and the four lines
> are not.

In [5]:
#@title 🔒 Reference implementation — later sections use `y`, `m`, `alpha_m`, `beta`, `idio`
y      = brk['BRK'] - brk['RF']                  # excess return (pitfall 1)
mkt_ex = brk['Mkt-RF']                           # the market, on the same months
X      = sm.add_constant(mkt_ex)                 # intercept (pitfall 2)
m      = sm.OLS(y, X).fit()

alpha_m, beta = m.params['const'], m.params['Mkt-RF']
idio = m.resid.std() * np.sqrt(12)

print(f"Berkshire on the market, {len(y)} months\n")
print(f"  mean excess return {y.mean()*12:+.2%}/yr   vol {y.std()*np.sqrt(12):.2%}/yr")
print(f"  alpha   {alpha_m*12:+.2%}/yr     t = {m.tvalues['const']:.2f}")
print(f"  beta    {beta:.2f}              t = {m.tvalues['Mkt-RF']:.1f}")
print(f"  R²      {m.rsquared:.2f}")
print(f"  idio vol {idio:.2%}/yr")

Berkshire on the market, 279 months

  mean excess return +12.51%/yr   vol 19.52%/yr
  alpha   +6.54%/yr     t = 1.80
  beta    0.59              t = 8.9
  R²      0.22
  idio vol 17.21%/yr


### Step 3 — Validate, then read it

Check the pitfalls: excess returns on the left ✅, `const` present ✅, both
series monthly ✅, α annualized by ×12 ✅.

Now read it.

> **🤔 Before I say anything — what does β = 0.59 tell you?**

**Berkshire was not a levered bet on the market. It was a defensive one.** When
the market moved a point, Berkshire moved about six tenths of one.

And yet it was *more* volatile than the market: 19.5% against 15.6%. Less market
exposure, more total risk. Whatever that extra risk is, it is not the index.
Let's split it.

---

## 3. Decomposing Risk, Hedging, and the Risk Budget <a id="decomp"></a>

Because $\varepsilon$ is uncorrelated with the factor by construction, variance
splits cleanly in two:

$$\underbrace{\sigma^2}_{\text{total}} = \underbrace{\beta^2\sigma_m^2}_{\text{systematic}} + \underbrace{\sigma_\varepsilon^2}_{\text{idiosyncratic}}$$

Note it's *variances* that add, not volatilities — so the two pieces combine
like the sides of a right triangle, not like a sum.

In [6]:
tot_vol = y.std() * np.sqrt(12)
sys_vol = beta * mkt_ex.std() * np.sqrt(12)

print(f"Berkshire total volatility {tot_vol:6.2%}")
print(f"  systematic (beta*mkt)    {sys_vol:6.2%}")
print(f"  idiosyncratic            {idio:6.2%}")
print(f"  check: sqrt({sys_vol:.4f}^2 + {idio:.4f}^2) = {np.sqrt(sys_vol**2 + idio**2):.2%}\n")
print(f"share of variance from the market (= R²): {m.rsquared:.0%}")
print(f"share idiosyncratic:                      {1-m.rsquared:.0%}")

Berkshire total volatility 19.52%
  systematic (beta*mkt)     9.21%
  idiosyncratic            17.21%
  check: sqrt(0.0921^2 + 0.1721^2) = 19.52%

share of variance from the market (= R²): 22%
share idiosyncratic:                      78%


> **💡 Key Insight: the volatility was real, and it was not the market**
>
> Berkshire ran 19.5% volatility against the market's 15.6%, with a beta of
> **0.59**. Only **9.2%** of that volatility was market exposure; **17.2%** was
> Berkshire's own. Roughly four fifths of its variance had nothing to do with
> the index.
>
> That is what a stock-picking record looks like in a regression: the risk sits
> in the picks, not in the exposure.

> **⚠️ Caution: R² is not a quality measure**
>
> Berkshire's R² is 0.22. That is not "22% good." It says 22% of Berkshire's
> variance was market movement. A *higher* R² would mean the returns on the factor explain more of the variation in the returns of the asset

### Why anyone bothers: the hedged portfolio

The decomposition isn't just accounting. It tells you how to *build* something.

Hold Berkshire and short $\beta$ units of the market. The systematic term
cancels:

$$r^e - \beta r^e_m = \alpha + \varepsilon$$

You are left with alpha plus idiosyncratic noise, and **no market exposure at
all**. This is the hedged portfolio, and its volatility is $\sigma_\varepsilon$ —
17.2% instead of Berkshire's 19.5%.

Now suppose your CIO gives you a **risk budget**: you may run \$5M of annualized
volatility in this position, no more. How much Berkshire can you hold?

In [7]:
BUDGET = 5_000_000       # $ of annualized volatility you're allowed to run

pos_unhedged = BUDGET / tot_vol      # $ position that uses the whole budget
pos_hedged   = BUDGET / idio

print(f"{'':22s}{'volatility':>12s}{'position':>16s}{'alpha P&L':>13s}")
print("-"*64)
for name, vol, pos in [('Berkshire, unhedged', tot_vol, pos_unhedged),
                       ('Berkshire, market-hedged', idio, pos_hedged)]:
    print(f"{name:22s}{vol:>12.2%}{pos:>16,.0f}{pos*alpha_m*12:>13,.0f}")

print(f"\nHedging lets you hold {pos_hedged/pos_unhedged-1:.0%} more Berkshire for the same risk budget,")
print(f"and therefore earn {pos_hedged/pos_unhedged-1:.0%} more alpha dollars.")

                        volatility        position    alpha P&L
----------------------------------------------------------------
Berkshire, unhedged         19.52%      25,618,005    1,674,319
Berkshire, market-hedged      17.21%      29,055,196    1,898,964

Hedging lets you hold 13% more Berkshire for the same risk budget,
and therefore earn 13% more alpha dollars.


> **💡 Key Insight: hedging converts risk budget into alpha capacity**
>
> Your risk budget is scarce. Every unit of it spent on market exposure — which
> you could have bought for five basis points — is a unit not spent on the thing
> you are actually being paid for.
>
> Strip the market out and the same budget carries a **13% larger** position in
> the part you have a view on. That is the entire economic case for hedging, and
> it is why the appraisal ratio (next section) is the number that matters when
> you already own the market.
>
> The gain is modest here *because* Berkshire carried so little market risk to
> begin with — there was not much to strip. Run the same arithmetic on a position
> with β near 1 and R² near 0.6, and the same budget carries 50% more.

> **⚠️ Caution: hedging does not always improve your Sharpe ratio**
>
> Look at what you gave up. Hedging removes $\beta E[f]$ from the numerator as
> well as $\beta^2\sigma_m^2$ from the denominator. If the factor premium is
> large and your α is small, you can easily end up with a *worse* Sharpe ratio
> than you started with.
>
> So why does anyone do it? Because exposure to a **factor** and exposure to
> **ε** are not the same kind of thing, even when the arithmetic looks alike.

> **📌 Remember**
>
> Anyone charging an alpha fee for something with β = 1.0 and α = 0 is selling
> you beta at an alpha price. A regression is how you catch them — which is
> exactly what we'll do next.

---

## 4. The Appraisal Ratio <a id="appraisal"></a>

Now the third question: **was any of it something you couldn't have bought
cheaply?**

You can buy market exposure for about five basis points. So the part of
Berkshire's return that came from β is not worth paying a manager for. What's left is α —
and the risk you had to carry to get it is the *idiosyncratic* risk, because the
systematic part was never yours in the first place.

$$AR = \frac{\alpha}{\sigma_\varepsilon}$$

That is the **appraisal ratio**: alpha per unit of the risk you couldn't
diversify or replicate.

In [8]:
sharpe_brk = y.mean()/y.std()*np.sqrt(12)
sharpe_mkt = mkt_ex.mean()/mkt_ex.std()*np.sqrt(12)      # the market, same months

print(f"{'':22s}{'Sharpe':>9s}{'Appraisal':>12s}")
print("-"*43)
print(f"{'Berkshire':22s}{sharpe_brk:>9.3f}{alpha_m*12/idio:>12.3f}")
print(f"{'The market':22s}{sharpe_mkt:>9.3f}{0.0:>12.3f}   <- by definition")

                         Sharpe   Appraisal
-------------------------------------------
Berkshire                 0.641       0.380
The market                0.649       0.000   <- by definition


### Why the appraisal ratio is lower than the Sharpe ratio

Berkshire's Sharpe is **0.641**. The market's, over the same months, is
**0.649**. On that measure Berkshire is the index.

Its appraisal ratio is **0.380**. The Sharpe ratio credits Berkshire for
*everything* it earned, including the 0.59 of market exposure anyone can buy for
five basis points. The appraisal ratio credits it only for α — and α is the part
where Berkshire was not the index at all.

### They were never three ratios

Go back to the endogenous benchmark from Section 2 — the fitted value
$r^b = \sum_j \beta_j f_j$. Measure the information ratio against *that*:

$$r - r^b = \alpha + \varepsilon
\qquad\Longrightarrow\qquad
IR = \frac{\text{mean}(\alpha + \varepsilon)}{\text{sd}(\alpha + \varepsilon)} = \frac{\alpha}{\sigma_\varepsilon}$$

which **is** the appraisal ratio, exactly.

So there is one formula — active return over tracking error — and the only thing
that changes is what you benchmark against:

| | = information ratio against… | Use when |
|---|---|---|
| **Sharpe** | cash | this is your whole portfolio |
| **Information ratio** | your mandate | you were hired to beat something |
| **Appraisal ratio** | **the endogenous benchmark** | you already hold the factors |

> **📌 Remember**
>
> Picking a performance measure *is* picking a benchmark. Sharpe assumes your
> alternative was a T-bill. The appraisal ratio assumes your alternative was a
> cheap replicating portfolio of factors — which, for anyone who can buy ETFs,
> is the honest comparison.

### And it can reverse the ranking

Here is value under the standard convention from Assignment 2 , so **+5.2%/yr and not the +20.8% of the sorts
lecture's demo** — run through the same regression.

In [9]:
q = dv[dv.exchcd == 1].groupby('date')['BM_l1'].quantile([.1,.9]).unstack().rename(
        columns={0.1:'lo', 0.9:'hi'})
dl = panel.merge(bm, on=['permno','date'], how='left').sort_values(['permno','date'])
dl['BM_l1'] = dl.groupby('permno')['BM'].shift(1)
dl['me_l1'] = dl.groupby('permno')['me'].shift(1)
dl = dl.dropna(subset=['BM_l1', 'ret', 'me_l1']).merge(q, on='date')
dl['g'] = np.where(dl.BM_l1 <= dl.lo, 0, np.where(dl.BM_l1 >= dl.hi, 9, np.nan))
dl = dl.dropna(subset=['g'])
pp = dl.groupby(['date','g']).apply(lambda g: np.average(g['ret'], weights=g['me_l1'])).unstack()
ls = (pp[9] - pp[0]).dropna()      # already dated by the month earned — no shift

j  = pd.concat([ls.rename('ls'), ff['Mkt-RF']], axis=1).dropna()
m2 = sm.OLS(j.ls, sm.add_constant(j['Mkt-RF'])).fit()
idio2 = m2.resid.std()*np.sqrt(12)

print(f"Value long-short (NYSE breakpoints, value-weighted)\n")
print(f"  mean      {j.ls.mean()*12:+.2%}/yr")
print(f"  alpha     {m2.params['const']*12:+.2%}/yr   t = {m2.tvalues['const']:.2f}")
print(f"  beta      {m2.params['Mkt-RF']:+.2f}")
print(f"  R²        {m2.rsquared:.3f}\n")
print(f"{'':22s}{'Sharpe':>9s}{'Appraisal':>12s}")
print("-"*43)
print(f"{'Berkshire':22s}{sharpe_brk:>9.3f}{alpha_m*12/idio:>12.3f}")
print(f"{'Value long-short':22s}{j.ls.mean()/j.ls.std()*np.sqrt(12):>9.3f}"
      f"{m2.params['const']*12/idio2:>12.3f}")

Value long-short (NYSE breakpoints, value-weighted)

  mean      +5.24%/yr
  alpha     +7.12%/yr   t = 2.44
  beta      -0.20
  R²        0.056

                         Sharpe   Appraisal
-------------------------------------------
Berkshire                 0.641       0.380
Value long-short          0.388       0.542


### The two ratios disagree

**Sharpe ranks Berkshire above the value strategy. Appraisal ranks the value
strategy above Berkshire.**

The reason is in the beta. The value long-short has β = **−0.20** — it leans
slightly *against* the market. That negative exposure dragged on its raw return
during a twenty-year bull market, so its Sharpe looks mediocre. Strip the market
out and its alpha (**+7.1%/yr**) is actually *larger* than its raw return
(+5.2%/yr).

> **💡 Key Insight**
>
> For an investor who already owns the market, the appraisal ratio is the
> relevant number, and it says the value strategy is the better addition. For an
> investor choosing one single thing to own, Sharpe is right and Berkshire wins.
>
> **The ratios don't disagree because one is wrong. They answer different
> questions.** Know which one you're asking.

> **⚠️ Mind the samples.** Berkshire's months are 1988–2021; the value long-short
> is 1980–2000. This compares two ratios, not two things you could have held
> side by side.

### Why the appraisal ratio is the number to care about?

If you already hold the market, the highest Sharpe ratio you can reach by adding
a fund to it, in the best proportion, is

$$SR_{\max} = \sqrt{SR_m^2 + AR^2}$$

The fund's own Sharpe ratio is not in the formula. Only the part of its return
the market can't replicate helps you, and the appraisal ratio is that part per
unit of risk.

For Berkshire: the market's Sharpe over its months was 0.649 and Berkshire's
appraisal ratio 0.380, so the best you could have done holding both is
√(0.649² + 0.380²) = **0.752**. Berkshire's own Sharpe ratio, 0.641, never
enters the calculation.

### Appraisal, t-statistic, and sample size

The appraisal ratio says how good the manager was. It says nothing about how
sure you can be, and that is a different number:

$$t(\alpha) \approx AR \times \sqrt{T} \qquad (T \text{ in years})$$

Certainty grows only with the square root of time. At an appraisal ratio of 0.4,
reaching t = 2 takes (2/0.4)² = **25 years** of data.

In [10]:
yrs = len(y) / 12
ar  = alpha_m * 12 / idio

print(f"appraisal ratio            {ar:.3f}")
print(f"years of data              {yrs:.1f}")
print(f"AR x sqrt(years)           {ar*np.sqrt(yrs):.2f}   <- the approximation")
print(f"t from the regression      {m.tvalues['const']:.2f}\n")
print(f"market Sharpe, same months {sharpe_mkt:.3f}")
print(f"best Sharpe holding both   {np.sqrt(sharpe_mkt**2 + ar**2):.3f}   = sqrt(SR_m^2 + AR^2)")

appraisal ratio            0.380
years of data              23.2
AR x sqrt(years)           1.83   <- the approximation
t from the regression      1.80

market Sharpe, same months 0.649
best Sharpe holding both   0.752   = sqrt(SR_m^2 + AR^2)


> **⚠️ Caution: three things we are not doing today**
>
> Berkshire's alpha has **t = 1.80** over 23 years — it does not clear |t| > 2,
> and we have not asked how many managers you would have to look at before one
> looked this good by chance. We also picked Berkshire *because* it is famous.
>
> The file's 279 months are also spread across a 398-month span, so about 30% of
> the calendar is missing.
>
> The first two are serious problems, and both are Lectures 8 and 9.

---

## 🛠️ Hands-On: Does Your Signal Have Alpha? <a id="ho1"></a>

In the sorts lecture you each picked a signal. The cell below builds its
long-short under the standard convention from Assignment 2 --
value-weighted. Regress it on the market and find out how much of it was market
exposure.

> **🤔 Predict first.** A long-short is roughly market-neutral by construction —
> you're long some stocks and short others. So do you expect β near zero? Commit
> before running.

In [11]:
#@title list of signals
menu  = pd.read_csv(f"{BASE}/signal_menu.csv")

print(f"menu : {len(menu)} signals available\n")
menu[['Acronym', 'Authors', 'Year', 'Cat.Economic']].head(30)

menu : 30 signals available



,Acronym,Authors,Year,Cat.Economic
0,OrgCap,Eisfeldt and Papanikolaou,2013,R&D
1,Accruals,Sloan,1996,accruals
2,NOA,Hirshleifer et al.,2004,asset composition
3,OScore,Dichev,1998,default risk
4,AnnouncementReturn,"Chan, Jegadeesh and Lakonishok",1996,earnings event
5,ShareIss5Y,Daniel and Titman,2006,external financing
6,ShareIss1Y,Pontiff and Woodgate,2008,external financing
7,CompositeDebtIssuance,"Lyandres, Sun and Zhang",2008,external financing
8,AssetGrowth,"Cooper, Gulen and Schill",2008,investment
9,InvestPPEInv,"Lyandres, Sun and Zhang",2008,investment


In [12]:
# === EDIT THIS CELL: your signal from Lecture 3 ===
MY_SIGNAL =       # ← your pick

sig = pd.read_parquet(f"{BASE}/signals/{MY_SIGNAL}.parquet")
x = panel.merge(sig, on=['permno','date'], how='left').sort_values(['permno','date'])
x['sig_l1'] = x.groupby('permno')[MY_SIGNAL].shift(1)
x['me_l1'] = x.groupby('permno')['me'].shift(1)
x = x.dropna(subset=['sig_l1', 'ret', 'me_l1'])
qq = (x[x.exchcd == 1].groupby('date')['sig_l1'].quantile([.1,.9]).unstack()
        .rename(columns={0.1:'lo', 0.9:'hi'}))
x = x.merge(qq, on='date')
x['g'] = np.where(x.sig_l1 <= x.lo, 0, np.where(x.sig_l1 >= x.hi, 9, np.nan))
x = x.dropna(subset=['g'])
px = x.groupby(['date','g']).apply(lambda g: np.average(g['ret'], weights=g['me_l1'])).unstack()
r  = (px[9] - px[0]).dropna()      # already dated by the month earned — no shift

SyntaxError: invalid syntax (2962837697.py, line 2)

In [ ]:
# === YOUR TURN ===
# Fill in the two blanks to regress your long-short on the market.

jj = pd.concat([r.rename('r'), ff['Mkt-RF']], axis=1).dropna()

mm =sm.OLS(jj.r, sm.add_constant(jj['Mkt-RF'])).fit()
iv = mm.resid.std() * np.sqrt(12)

print(f"{MY_SIGNAL}")
print(f"  raw return  {jj.r.mean()*12:+7.2%}/yr    Sharpe    {jj.r.mean()/jj.r.std()*np.sqrt(12):.2f}")
print(f"  alpha       {mm.params['const']*12:+7.2%}/yr    t = {mm.tvalues['const']:.2f}")
print(f"  beta        {mm.params['Mkt-RF']:+7.2f}")
print(f"  R²          {mm.rsquared:7.3f}")
print(f"  appraisal   {mm.params['const']*12/iv:+7.2f}")

### Compare with the room

Two things to look for, and to say out loud:

- **Is your β near zero?** Most long-shorts are, but not all. A signal
  correlated with size or volatility often carries real market exposure without
  anyone intending it.
- **Is your alpha bigger or smaller than your raw return?** Bigger means
  negative beta was *hurting* you. Smaller means part of what looked like signal
  was just market.

---

## 🎯 Challenge: Two Factors and the Market <a id="challenge"></a>

*Homework — due before Lecture 5. Graded on completion.*

Pick **two** signals from the menu and build a long-short for each, exactly as
the Hands-On did: signal lagged one month, NYSE breakpoints, value-weighted, top
decile minus bottom. Pick two you expect to be different in kind — an accounting
signal and a price-based one, say.

Everything below is measured against the market, `ff['Mkt-RF']`, over the months
your series and the market share.

> **⚠️** `DivSeason`, `OScore` and `ShareIss1Y` take too few distinct values to
> cut cleanly. Pick anything else.

### Q1 — Each factor on its own

Regress each long-short on the market and report, for **each** factor: the
annualized alpha, the beta, the appraisal ratio, the factor's own Sharpe ratio,
and its information ratio against the market.

Three of those five you have seen computed today; two you have to think about.
The Sharpe ratio of a long-short has no risk-free rate to subtract. The
information ratio is the mean of `factor − market` over its standard deviation,
annualized.

> **📌 Required variable names:**
> ```python
> f1 = "____"          # the acronym you picked, e.g. "GP"
> f2 = "____"
>
> f1_alpha     = ____  # annualized, 0.05 = 5%/yr
> f1_beta      = ____
> f1_appraisal = ____  # annualized alpha / annualized idiosyncratic vol
> f1_sharpe    = ____  # the factor's own Sharpe ratio
> f1_ir        = ____  # information ratio against the market
>
> f2_alpha     = ____
> f2_beta      = ____
> f2_appraisal = ____
> f2_sharpe    = ____
> f2_ir        = ____
> ```

In [16]:
f1, f2 = "GP", "Mom12m"

# Accounting signal: GP
sig = pd.read_parquet(f"{BASE}/signals/GP.parquet")
x = panel.merge(sig, on=["permno", "date"], how="left").sort_values(["permno", "date"])
x["sig_l1"] = x.groupby("permno")["GP"].shift(1)
x["me_l1"] = x.groupby("permno")["me"].shift(1)
x = x.dropna(subset=["sig_l1", "ret", "me_l1"])
cuts = x[x.exchcd == 1].groupby("date")["sig_l1"].quantile([.1, .9]).unstack()
cuts.columns = ["lo", "hi"]
x = x.join(cuts, on="date")
high = x[x.sig_l1 >= x.hi].groupby("date").apply(lambda g: np.average(g.ret, weights=g.me_l1))
low = x[x.sig_l1 <= x.lo].groupby("date").apply(lambda g: np.average(g.ret, weights=g.me_l1))
GP = high - low

# Price signal: Mom12m
sig = pd.read_parquet(f"{BASE}/signals/Mom12m.parquet")
x = panel.merge(sig, on=["permno", "date"], how="left").sort_values(["permno", "date"])
x["sig_l1"] = x.groupby("permno")["Mom12m"].shift(1)
x["me_l1"] = x.groupby("permno")["me"].shift(1)
x = x.dropna(subset=["sig_l1", "ret", "me_l1"])
cuts = x[x.exchcd == 1].groupby("date")["sig_l1"].quantile([.1, .9]).unstack()
cuts.columns = ["lo", "hi"]
x = x.join(cuts, on="date")
high = x[x.sig_l1 >= x.hi].groupby("date").apply(lambda g: np.average(g.ret, weights=g.me_l1))
low = x[x.sig_l1 <= x.lo].groupby("date").apply(lambda g: np.average(g.ret, weights=g.me_l1))
Mom12m = high - low

returns = pd.concat([GP.rename(f1), Mom12m.rename(f2), ff["Mkt-RF"]], axis=1).dropna()
market = returns["Mkt-RF"]
print(f"Common sample: {len(returns)} months, {returns.index.min():%Y-%m} to {returns.index.max():%Y-%m}")

r = returns["GP"]
model = sm.OLS(r, sm.add_constant(market)).fit()
f1_alpha = model.params["const"] * 12
f1_beta = model.params["Mkt-RF"]
f1_appraisal = f1_alpha / (model.resid.std() * np.sqrt(12))
f1_sharpe = r.mean() / r.std() * np.sqrt(12)
active = r - market
f1_ir = active.mean() / active.std() * np.sqrt(12)
print(f"GP: alpha {f1_alpha:.2%}, beta {f1_beta:.4f}, AR {f1_appraisal:.4f}, Sharpe {f1_sharpe:.4f}, IR {f1_ir:.4f}")

r = returns["Mom12m"]
model = sm.OLS(r, sm.add_constant(market)).fit()
f2_alpha = model.params["const"] * 12
f2_beta = model.params["Mkt-RF"]
f2_appraisal = f2_alpha / (model.resid.std() * np.sqrt(12))
f2_sharpe = r.mean() / r.std() * np.sqrt(12)
active = r - market
f2_ir = active.mean() / active.std() * np.sqrt(12)
print(f"Mom12m: alpha {f2_alpha:.2%}, beta {f2_beta:.4f}, AR {f2_appraisal:.4f}, Sharpe {f2_sharpe:.4f}, IR {f2_ir:.4f}")


Common sample: 251 months, 1980-02 to 2000-12
GP: alpha 6.21%, beta 0.0423, AR 0.5701, Sharpe 0.6046, IR -0.1386
Mom12m: alpha 18.11%, beta 0.1880, AR 0.9069, Sharpe 0.9826, IR 0.4510


### Q2 — Each factor, added to the market

You already hold the market. Section 4 gives the best Sharpe ratio you can reach
by adding **one** more thing to it:

$$SR_{\max} = \sqrt{SR_m^2 + AR^2}$$

Do that once for each of your factors, and report the market's own Sharpe ratio
over the same months alongside them.

> **📌 Required variable names:**
> ```python
> sr_market = ____   # the market's Sharpe ratio, same months
> sr_max_f1 = ____   # best Sharpe from the market plus factor 1
> sr_max_f2 = ____   # best Sharpe from the market plus factor 2
> ```

In [17]:
sr_market = market.mean() / market.std() * np.sqrt(12)
sr_max_f1 = np.sqrt(sr_market**2 + f1_appraisal**2)
sr_max_f2 = np.sqrt(sr_market**2 + f2_appraisal**2)
print(f"Market alone: {sr_market:.4f}")
print(f"Market + {f1}: {sr_max_f1:.4f}")
print(f"Market + {f2}: {sr_max_f2:.4f}")

Market alone: 0.5866
Market + GP: 0.8180
Market + Mom12m: 1.0801


### Q3 — Both factors at once

**This one is harder than anything else today. Have a go at it.**

Forget the market for a moment. Hold **both** of your factors, in whatever
proportions are best. What is the highest Sharpe ratio that combination can
reach?

Use what we learned to get there!

> **📌 Required variable name:**
> ```python
> sr_both = ____     # best annualized Sharpe from the two factors together
> ```

In [18]:
# Treat GP as the benchmark and add Mom12m.
fit = sm.OLS(returns[f2], sm.add_constant(returns[f1])).fit()
ar_extra = fit.params["const"] / fit.resid.std() * np.sqrt(12)
sr_both = np.sqrt(f1_sharpe**2 + ar_extra**2)
print(f"{f1} + {f2}: best Sharpe {sr_both:.4f}")

# Check against the mean-covariance solution.
factors = returns[[f1, f2]]
w = np.linalg.solve(factors.cov(), factors.mean())
combined = factors @ w
assert np.isclose(sr_both, combined.mean() / combined.std() * np.sqrt(12))
assert sr_both >= max(f1_sharpe, f2_sharpe)
print("Independent calculation agrees.")

GP + Mom12m: best Sharpe 1.1193
Independent calculation agrees.



### Q4 — The memo

> **📝 Maximum 6 sentences**
>
> You now know what each factor adds to the market on its own, and what the two
> of them can do together without it. How would you work out the best Sharpe
> ratio available from **all three** — the market and both factors — held in the
> best proportions?
>
> I want you to especulate about how to get this done.

In [19]:
MEMO = """
I would align the market, GP, and Mom12m to the same months and estimate their mean returns and covariance matrix.
I would choose weights to maximize the combined mean return divided by its volatility, allowing short positions as in the unconstrained calculations above.
The optimal weights are proportional to the inverse covariance matrix times the mean-return vector.
The maximum annualized Sharpe ratio is the square root of 12 times the mean vector transposed, multiplied by the inverse covariance matrix and then the mean vector.
This accounts for correlations among all three returns, so I would not simply add the two appraisal ratios squared to the market Sharpe squared.
I would test the estimated weights on later data and account for trading costs because the in-sample optimum can overstate achievable performance.
"""
print(MEMO)


I would align the market, GP, and Mom12m to the same months and estimate their mean returns and covariance matrix.
I would choose weights to maximize the combined mean return divided by its volatility, allowing short positions as in the unconstrained calculations above.
The optimal weights are proportional to the inverse covariance matrix times the mean-return vector.
The maximum annualized Sharpe ratio is the square root of 12 times the mean vector transposed, multiplied by the inverse covariance matrix and then the mean vector.
This accounts for correlations among all three returns, so I would not simply add the two appraisal ratios squared to the market Sharpe squared.
I would test the estimated weights on later data and account for trading costs because the in-sample optimum can overstate achievable performance.



---

## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["f1", "f2",
            "f1_alpha", "f1_beta", "f1_appraisal", "f1_sharpe", "f1_ir",
            "f2_alpha", "f2_beta", "f2_appraisal", "f2_sharpe", "f2_ir",
            "sr_market", "sr_max_f1", "sr_max_f2", "sr_both", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L4_PerfEval_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "factors": [f1, f2],
    "answers": {k: float(eval(k)) for k in required if k not in ("f1", "f2", "MEMO")},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **Three ratios, one formula:** active return over tracking error, against
   cash (Sharpe), a mandate (information), or what you could have bought
   cheaply (appraisal).

2. **The benchmark is a choice, and it manufactures performance.** A value
   manager measured against the market gets paid for the tilt.

3. **α is what the factor doesn't explain; β is what you could have bought
   cheaply.**

4. **Variances add, volatilities don't:** σ² = β²σ²ₘ + σ²ε.

5. **A factor model does two jobs:** risk (β, σε) and expected return (α). A
   high R² helps the first and leaves less room for the second.

6. **Berkshire was skill, not leverage.** β = 0.59, α = +6.5%/yr, and four
   fifths of its variance was idiosyncratic.

7. **Hedging converts risk budget into alpha capacity.** Beta is cheap and alpha
   is scarce, so they should cost different amounts.

8. **Sharpe and appraisal can disagree.** Berkshire beats the value long-short
   on Sharpe; value beats Berkshire on appraisal, because its β is negative.
   Which one matters depends on what you already own.

9. **The appraisal ratio sets how far a fund can raise your Sharpe:**
   SR²max = SR²ₘ + AR².

10. **t ≈ AR × √T.** Berkshire's 0.38 over 23 years gives t = 1.8; an appraisal
    ratio of 0.4 needs about 25 years to reach t = 2.

---

### Next class

Where do factors come from? We used the market because it was obvious. Next:
the other factors people use, where they came from, and how you'd choose.

---

## 📎 Appendix <a id="appendix"></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📎 APPENDIX — Belt-and-Suspenders Data Loading
# ═══════════════════════════════════════════════════════════════════════
#   panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")
#
# Fama-French factors are fetched live from Ken French's library. Prompt:
# "Using pandas-datareader, fetch F-F_Research_Data_Factors monthly from the
#  famafrench source starting 1980, convert the PeriodIndex to month-end
#  timestamps, and divide by 100 to get decimals."
def fetch_ff_monthly():
    import pandas_datareader.data as web
    f = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-01-01')[0]
    f.index = pd.to_datetime(f.index.to_timestamp()) + pd.offsets.MonthEnd(0)
    return f / 100

# Backup if Ken French is unreachable:
# ff = pd.read_csv("https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/ff_monthly.csv", index_col=0, parse_dates=True)
